# NBA Role Clustering

This notebook builds an unsupervised clustering model over recent NBA player seasons to discover **playstyle roles**, using only statistical profiles (not listed positions).

Roles are later interpreted in terms of your modern NBA role framework (Offensive Hub, Movement Shooter, Stretch Big, Rim Runner, Connector, etc.).

In [ ]:
# Imports and configuration

import os

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.data_fetch import cache_player_season_table
from src.features import build_feature_matrix
from src.clustering import search_kmeans_k, run_kmeans, nearest_examples_to_centroids
from src.roles import summarize_clusters, assign_role_labels, role_summary_table, example_players_by_role

sns.set_theme(style="darkgrid")

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)

PLAYER_SEASON_PATH = os.path.join(DATA_DIR, "player_season_stats.parquet")

print("Data directory:", os.path.abspath(DATA_DIR))

## 1. Fetch and cache player-season data

This step:

- Uses `nba_api` to pull per-player per-season **base + advanced + usage + scoring** stats for recent seasons.
- Adds height/weight via the player bio endpoint.
- Optionally adds clutch/on-off style impact proxies.
- Caches the combined table to `data/player_season_stats.parquet`.

You only need to run this cell occasionally; after that, you can reload from disk.

In [ ]:
# This cell may take a while due to API rate limits.

from src.data_fetch import build_player_season_table

seasons = ("2022-23", "2023-24")

if not os.path.exists(PLAYER_SEASON_PATH):
    print("Fetching player-season table for seasons:", seasons)
    df_raw = build_player_season_table(seasons=seasons, active_only=True)
    df_raw.to_parquet(PLAYER_SEASON_PATH, index=False)
    print("Saved to", PLAYER_SEASON_PATH)
else:
    print("Cached file already exists at", PLAYER_SEASON_PATH)

# Quick peek at the raw table
raw = pd.read_parquet(PLAYER_SEASON_PATH)
raw.head()

## 2. Build feature matrix (no listed positions)

We engineer features aligned with the role framework:

- **Offensive load & playmaking**: USG%, AST%, TOV%, AST/TOV.
- **Efficiency & scoring**: TS%, 3PA rate, FT rate, scoring per minute.
- **Rebounding & defense**: OREB/DREB/REB per minute, STL/BLK per minute.
- **Impact proxies**: plus-minus and offensive/defensive/net rating (including any clutch variants).

We also add height/weight but **drop official listed positions from the model features** (they're kept only for later interpretation).

In [ ]:
feats, feature_cols = build_feature_matrix(
    path=PLAYER_SEASON_PATH,
    min_minutes=600.0,
    min_games=25,
)

print("Feature columns (used for clustering):", len(feature_cols))
feature_cols

## 3. Exploratory data analysis

We quickly explore the distribution and correlation structure of the features to get intuition for how roles might separate in this feature space.

In [ ]:
# Basic summary
feats.describe(include="all").T.head(30)

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = feats[feature_cols].corr()
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()

## 4. K-Means model selection

We run K-Means for a range of `k` and inspect the silhouette scores. You can treat this as a starting point, then choose `k` based on both scores and how interpretable the resulting clusters are in terms of the role framework.

In [ ]:
k_values = list(range(8, 25))
search_df = search_kmeans_k(feats, feature_cols, k_values=k_values)
search_df

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(search_df["k"], search_df["silhouette"], marker="o")
plt.xlabel("k (number of clusters)")
plt.ylabel("Silhouette score")
plt.title("K-Means model selection")
plt.grid(True)
plt.show()

In [ ]:
# Choose the best k by silhouette (you can override manually).
best_row = search_df.sort_values("silhouette", ascending=False).iloc[0]
BEST_K = int(best_row["k"])
print("Best k by silhouette:", BEST_K, "(score =", round(best_row["silhouette"], 3), ")")

## 5. Fit final K-Means model and inspect clusters

We fit K-Means with the chosen `k` and look at:

- Cluster-level feature means.
- Example players closest to each cluster centroid.

This is still **positionless** in terms of the input features; we only use listed position for later sanity checks, not for clustering.

In [ ]:
feats_clustered, km, scaler, sil = run_kmeans(feats, feature_cols, k=BEST_K)
print("Final silhouette score:", round(sil, 3))

cluster_means, cluster_sizes = summarize_clusters(feats_clustered, feature_cols)
cluster_sizes

In [ ]:
cluster_means.round(3)

In [ ]:
examples_by_cluster = nearest_examples_to_centroids(
    feats_clustered,
    feature_cols,
    kmeans=km,
    scaler=scaler,
    n_examples=10,
)

# Peek at example players for a few clusters
for cid in sorted(examples_by_cluster.keys())[:5]:
    print("\n=== Cluster", cid, "===")
    display(examples_by_cluster[cid].head(10))

## 6. Map clusters to role labels

Here you manually map each numeric cluster ID to one of the roles from the framework.

Start with heuristics like:

- High USG% + AST% + balanced shot profile → **Offensive Hub / Do-It-All Engine** or **Three-Level Shot Creator**.
- High AST% + lower USG% → **Pass-First Orchestrator**.
- Very high 3PA rate + low USG% → **Spot-Up Specialist** or **Movement Shooter**.
- High OREB/DREB per minute + low 3PA rate + high FT rate → **Rim Runner / Vertical Spacer** or **Rebounding Specialist**.
- High STL/BLK per minute + strong defensive/net rating → **Point-of-Attack Defender**, **Switchable Wing Defender**, or **Rim Protector**.

You can refine these mappings iteratively as you inspect cluster summaries and example players.

In [ ]:
# Example: start with an empty mapping, then fill in once you inspect clusters.

cluster_to_role = {
    # 0: "Offensive Hub / Do-It-All Engine",
    # 1: "Pass-First Orchestrator",
    # 2: "Movement Shooter",
    # ...
}

feats_with_roles = assign_role_labels(feats_clustered, cluster_to_role)
feats_with_roles.head()

## 7. Role-level summaries and example players

Once `cluster_to_role` is filled in, we can summarize each role and pull example players.

In [ ]:
role_summary = role_summary_table(feats_with_roles, feature_cols)
role_summary

In [ ]:
examples_by_role = example_players_by_role(feats_with_roles, n=12)

for role, df_role in examples_by_role.items():
    print("\n=== Role:", role, "===")
    display(df_role)

## 8. Interpreting a real run

When you run this notebook end-to-end on current data:

1. **Check model quality**
   - Is the silhouette score reasonably high for your chosen `k`?
   - Do cluster sizes look reasonable (no huge "everything" cluster or many singletons)?

2. **Validate clusters vs. intuition**
   - For each cluster, skim the `cluster_means` row and example players.
   - Ask whether the statistical profile and names match one of your desired roles.
   - If a cluster mixes roles (e.g., shooters and slashers together), tweak features or increase `k`.

3. **Iterate on features and K**
   - Add/remove features that clarify distinctions (e.g., emphasize rebounding for bigs, or usage for on-ball engines).
   - Try separate runs for different height bands if you want more granularity among guards/wings/bigs while still ignoring official positions.

4. **Optional: add richer data**
   - Integrate play-type frequency tables (PnR ball handler, roll man, spot up, etc.).
   - Join external impact metrics (EPM, RAPM, RAPTOR) or lineup-based on/off data.

Over time, you can use this notebook as a sandbox to refine how **data-driven roles** emerge relative to your theoretical framework.